# RouteIQ — Exploratory Data Analysis
Analyzes 50k sample delivery records to validate data quality and surface operational insights.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sys
sys.path.insert(0, '..')

orders = pd.read_parquet('../data/sample/orders_enriched.parquet')
riders = pd.read_parquet('../data/sample/riders.parquet')
stores = pd.read_parquet('../data/sample/dark_stores.parquet')
print(f'Orders: {len(orders):,} rows, {len(orders.columns)} features')
orders.head()

In [ ]:
# Key operational metrics
print('=== Operational Summary ===')
print(f'Late delivery rate:    {orders["is_late"].mean():.1%}')
print(f'Avg delivery time:     {orders["actual_delivery_min"].mean():.1f} min')
print(f'Avg delay (late only): {orders[orders["is_late"]==1]["delay_minutes"].mean():.1f} min')
print(f'Avg cost per order:    ₹{orders["total_cost_inr"].mean():.2f}')
print(f'Peak hour orders:      {orders["is_peak_hour"].mean():.1%}')
print(f'Weather impacted:      {(orders["weather_delay_min"]>5).mean():.1%}')

In [ ]:
# Hourly demand pattern
hourly = orders.groupby('hour_of_day').size().reset_index(name='orders')
fig = px.bar(hourly, x='hour_of_day', y='orders',
             title='Order Volume by Hour of Day',
             color='orders', color_continuous_scale='Blues')
fig.show()

In [ ]:
# City-level late rate
city_kpis = orders.groupby('city').agg(
    orders=('order_id','count'),
    late_rate=('is_late','mean'),
    avg_time=('actual_delivery_min','mean'),
    avg_cost=('total_cost_inr','mean'),
).reset_index()
city_kpis['late_pct'] = (city_kpis['late_rate']*100).round(1)
print(city_kpis.sort_values('late_pct', ascending=False).to_string(index=False))

In [ ]:
# Feature correlations with late delivery
num_cols = ['delivery_distance_km','congestion_factor','weather_delay_min',
            'route_efficiency_score','prep_time_min','is_peak_hour','experience_months']
corr = orders[num_cols + ['is_late']].corr()['is_late'].drop('is_late').sort_values(ascending=False)
print('Correlation with late delivery:')
print(corr.round(4))